In [151]:
import pandas as pd

In [152]:
gender_df_og = pd.read_excel("2024_gender_gen_election.xlsx")
age_df_og = pd.read_excel("2024_age_gen_election.xlsx")
race_df_og = pd.read_excel("2024_race_gen_election.xlsx")
total_df_og = pd.read_excel("2024_total_gen_election.xlsx")

In [153]:
gender_df = gender_df_og.copy()
age_df = age_df_og.copy()
race_df = race_df_og.copy()
total_df = total_df_og.copy()

In [164]:
# clean headers
for df in [gender_df, age_df, race_df, total_df]:
    df.columns.str.strip().str.replace("\n"," ")

gender_df = gender_df.rename(columns={"Total Ballots Cast": "Total Ballots Cast - Gender"})
age_df = age_df.rename(columns={"Total Ballots\nCast": "Total Ballots Cast - Age"})
race_df = race_df.rename(columns={"Total Ballots Cast": "Total Ballots Cast - Race"})
total_df = total_df.rename(columns={"Total Ballots Cast": "Total Ballots Cast - Total"})

In [165]:
for df in [gender_df, age_df, race_df, total_df]:
    df["County"] = df["County"].str.strip()
    df["County"] = df["County"].str.replace("\n"," ")
    df["County"] = df["County"].str.title()

In [166]:
# join the three dfs
merged = gender_df.merge(age_df, on="County").merge(race_df, on="County")

In [167]:
merged.columns.tolist()

['County',
 'Total Ballots Cast - Gender',
 'Total Female',
 'Total Male',
 'Total Not Identified',
 'Total Ballots Cast - Age',
 'Age 18-29',
 'Age 30-39',
 'Age 40-49',
 'Age 50-59',
 'Age 60-69',
 'Age 70-79',
 'Age 80-89',
 'Age 90-99',
 'Age 100+',
 'Total Ballots Cast - Race',
 'Total Asian (A)',
 'Total American Indian (AI)',
 'Total Black (B)',
 'Total Federally- Registered (may be of any race) (F)',
 'Total Hispanic (H)',
 'Total Korean (K)',
 'Total Other (O)',
 'Total White (W)',
 'Total Not Identified (U)']

In [168]:
merged["equal"] = (
    (merged["Total Ballots Cast - Gender"] == merged["Total Ballots Cast - Age"]) &
     (merged["Total Ballots Cast - Gender"] == merged["Total Ballots Cast - Race"])
)

mismatches = merged[~merged["equal"]]

In [169]:
print(mismatches)
# output: empty dataframe. means there are no mismatches in total ballots cast among the three dfs. 

Empty DataFrame
Columns: [County, Total Ballots Cast - Gender, Total Female, Total Male, Total Not Identified, Total Ballots Cast - Age, Age 18-29, Age 30-39, Age 40-49, Age 50-59, Age 60-69, Age 70-79, Age 80-89, Age 90-99, Age 100+, Total Ballots Cast - Race, Total Asian (A), Total American Indian (AI), Total Black (B), Total Federally- Registered (may be of any race) (F), Total Hispanic (H), Total Korean (K), Total Other (O), Total White (W), Total Not Identified (U), equal]
Index: []

[0 rows x 26 columns]


In [170]:
compare = merged.merge(total_df, on="County")

compare.head()

,County,Total Ballots Cast - Gender,Total Female,Total Male,Total Not Identified,Total Ballots Cast - Age,Age 18-29,Age 30-39,Age 40-49,Age 50-59,...,Total White (W),Total Not Identified (U),equal,Registered Voters,Total Ballots Cast - Total,Democrat Ballots,Republican Ballots,Absentee Ballots,Turnout as a\nPercentage of Registered Voters,Absentee as\nPercentage of Total Ballots
0,Autauga,28401,15750,12581,82,28401,3600,3991,4631,5223,...,22571,76,True,46291,28388,4960,13269,1670,0.6133,0.0588
1,Baldwin,122501,66932,55452,117,122501,11800,13847,17274,20669,...,112966,260,True,207643,122542,14334,69308,7771,0.5902,0.0634
2,Barbour,9901,5646,4236,19,9901,970,949,1206,1670,...,5979,40,True,17666,9919,3409,2419,593,0.5615,0.0598
3,Bibb,9283,5017,4264,2,9283,1275,1211,1323,1737,...,7947,25,True,15152,9278,1245,4628,351,0.6123,0.0378
4,Blount,28255,15026,13213,16,28255,3914,3800,4195,5005,...,27068,47,True,43601,28138,1534,18524,965,0.6454,0.0343


In [ ]:
# get the difference between total ballots cast and total ballots cast gender (age, and race - remember the last three have the same values)
compare["difference"] = compare["Total Ballots Cast - Total"] - compare["Total Ballots Cast - Gender"]

In [174]:
compare["abs_difference"] = compare["difference"].abs()

In [199]:
compare["difference_percent"] = (compare["abs_difference"]/compare["Registered Voters"])

In [ ]:
# the highest difference_percent is 0.056314 so the difference in values between total ballots cast for gender, age, and total aren't too different from the total_df values for total ballots cast
compare.sort_values("difference_percent", ascending=False)[["County", "Total Ballots Cast - Gender", "Total Ballots Cast - Age", "Total Ballots Cast - Race", "Total Ballots Cast - Total", "Registered Voters", "difference_percent", "difference", "abs_difference"]].head()

,County,Total Ballots Cast - Gender,Total Ballots Cast - Age,Total Ballots Cast - Race,Total Ballots Cast - Total,Registered Voters,difference_percent,difference,abs_difference
17,Conecuh,5552,5552,5552,6105,9820,0.056314,553,553
6,Butler,8112,8112,8112,8530,14530,0.028768,418,418
8,Chambers,13687,13687,13687,14380,26794,0.025864,693,693
9,Cherokee,12528,12528,12528,13030,21383,0.023477,502,502
38,Lauderdale,42188,42188,42188,43816,70858,0.022976,1628,1628


In [188]:
compare.columns.tolist()

['County',
 'Total Ballots Cast - Gender',
 'Total Female',
 'Total Male',
 'Total Not Identified',
 'Total Ballots Cast - Age',
 'Age 18-29',
 'Age 30-39',
 'Age 40-49',
 'Age 50-59',
 'Age 60-69',
 'Age 70-79',
 'Age 80-89',
 'Age 90-99',
 'Age 100+',
 'Total Ballots Cast - Race',
 'Total Asian (A)',
 'Total American Indian (AI)',
 'Total Black (B)',
 'Total Federally- Registered (may be of any race) (F)',
 'Total Hispanic (H)',
 'Total Korean (K)',
 'Total Other (O)',
 'Total White (W)',
 'Total Not Identified (U)',
 'equal',
 'Registered Voters',
 'Total Ballots Cast - Total',
 'Democrat Ballots',
 'Republican Ballots',
 'Absentee Ballots',
 'Turnout as a\nPercentage of Registered Voters',
 'Absentee as\nPercentage of Total Ballots',
 'difference',
 'abs_difference']